In [1]:
import pandas as pd

In [2]:
# Load data
SQ_ref_proteins = pd.read_csv('../data/SQ_ref_enzymes.tsv', sep='\t')

meta_table = pd.read_csv('../data/SQ_metatable_all.tsv', sep='\t')

meta_table_merge = meta_table.merge(
    SQ_ref_proteins,
    left_on='query',
    right_on='Protein ID',
    how='left'
)

meta_table_merge['query'].unique()
# There is no ANG62941.1 (sqoD - sulfo-ASDO) in diamond hits

array(['EJF39090.1', 'WP_017967310.1', 'PRO65853.1', 'EJF39103.1',
       'PRO65852.1', 'A0A3A6N9T6.1', 'EJF39099.1', 'EJF39089.1',
       'MCD5344479.1', 'PRO65851.1', 'AYG21326.1', 'AAK90112.1',
       'PRO65854.1', 'WP_017967307.1', 'AYG21321.1', 'PRO65849.1',
       'AAK90113.2', 'WP_017967308.1', 'AYG21324.1', 'AYG21325.1',
       'AYG21322.1', 'A0A3A6NE59.1', 'A0A3A6N9V6.1', 'ADE70661.1',
       'WP_017967309.1', 'AYG21323.1', 'AAK90111.1'], dtype=object)

The table (meta_table_merge) contains rows with hypothetical proteins in target column that are duplicates (this is because bakta creates an additional faa file with only hypothetical proteins during annotation)

In [3]:
meta_table_filtered = meta_table_merge[~meta_table_merge['target'].str.contains('hypotheticals', case=False)].copy()

## SQ clusters search

In [4]:
# sorting and distance calculation
meta_table_sorted = meta_table_filtered.sort_values(['contig', 'start'])
meta_table_sorted['mid'] = (meta_table_sorted['start'] + meta_table_sorted['end']) / 2
meta_table_sorted['distance'] = meta_table_sorted.groupby('contig')['mid'].diff().abs()

# clusters identification
threshold = 5000
meta_table_sorted['cluster'] = (
    (meta_table_sorted['distance'] > threshold) |
    (meta_table_sorted['distance'].isna())
).cumsum()

# clusters with more than 1 gene
cluster_sizes = meta_table_sorted.groupby('cluster').size()
candidate_clusters = cluster_sizes[cluster_sizes > 1].index
candidates = meta_table_sorted[meta_table_sorted['cluster'].isin(candidate_clusters)].copy()
candidates['cluster_size'] = candidates.groupby('cluster')['cluster'].transform('size')

print(f"Cluster candidates (>1 gene): {len(candidate_clusters)}")
print(f"Total number of genes in clusters: {len(candidates)}")

Cluster candidates (>1 gene): 4271
Total number of genes in clusters: 9740


In [5]:
candidates.to_csv("../data/clusters_candidates.csv", index=False)

In [6]:
# filtering info
total_before = len(candidates.index)

df_q = candidates[
    (candidates['evalue'] < 1e-5) &
    (candidates['bitscore'] > 50)
].copy()

total_after = len(df_q)

dropped = total_before - total_after
dropped_percent = dropped / total_before * 100

print(f"Total before filtration: {total_before}")
print(f"Total after filtration: {total_after}")
print(f"Dropped: {dropped} ({dropped_percent:.1f}%)")

Total before filtration: 9740
Total after filtration: 9250
Dropped: 490 (5.0%)


In [7]:
# building loci table
loci = df_q.groupby('cluster').agg({
    'Gene': lambda x: ','.join(sorted(set(x.dropna())))
}).reset_index()

In [8]:
def annotate_sulfo_pathways(gene_list):
    genes = {g.lower() for g in gene_list}

    results = {}

    # upstream
    up_block1 = bool(genes & {"yihq", "sqga"})
    up_block2 = bool(genes & {"yihs", "sqvd"})
    n_up_blocks = sum([up_block1, up_block2])

    # downstream
    downstream_map = {
        "sulfo-EMP": {"yihu", "slab"},
        "sulfo-ED": {"yihu", "slab"},
        "sulfo-TAL": {"yihu", "slab"},
        "sulfo-TK": {"sqwk", "sqwl"},
        "sulfo-ASMO": {"squf"},
        "sulfo-ASDO": {"squf"},
    }

    # CORE SCORING
    core_info = {}

    # EMP
    emp_hits = len(genes & {"sqia", "yiht", "sqik", "yihv"})
    core_info["sulfo-EMP"] = {
        "score": emp_hits,
        "is_core": emp_hits >= 2,
        "is_strict": (
            bool(genes & {"sqia", "yiht"}) and
            bool(genes & {"sqik", "yihv"})
        )
    }

    # ED
    ed_core = {"seda", "sedb", "sedc", "sedd"}
    ed_hits = len(genes & ed_core)
    core_info["sulfo-ED"] = {
        "score": ed_hits,
        "is_core": ed_hits >= 3,
        "is_strict": ed_hits >= 3
    }

    # TK
    tk_hits = sum([
        {"sqwg", "sqwh"} <= genes,
        "sqwi" in genes,
        bool(genes & {"sqwf", "sqwd"})
    ])
    core_info["sulfo-TK"] = {
        "score": tk_hits,
        "is_core": tk_hits >= 3,
        "is_strict": tk_hits >= 3
    }

    # TAL
    core_info["sulfo-TAL"] = {
        "score": int("sqva" in genes),
        "is_core": "sqva" in genes,
        "is_strict": "sqva" in genes
    }

    # ASMO
    core_info["sulfo-ASMO"] = {
        "score": int("squd" in genes),
        "is_core": "squd" in genes,
        "is_strict": "squd" in genes
    }

    # ASDO
    core_info["sulfo-ASDO"] = {
        "score": int("sqod" in genes),
        "is_core": "sqod" in genes,
        "is_strict": "sqod" in genes
    }

    # FILTER: только реальные core
    core_hits = {
        p: v for p, v in core_info.items() if v["is_core"]
    }

    # SCORING
    for pathway, info in core_hits.items():

        has_up = n_up_blocks >= 1
        has_down = bool(genes & downstream_map[pathway])

        if has_up and has_down:
            score = "max"
        elif has_up:
            score = "min_up"
        elif has_down:
            score = "min_down"
        else:
            score = "min"

        results[pathway] = {
            "score": score,
            "core_score": info["score"],
            "n_up": n_up_blocks
        }

    # hybrid check
    n_core_paths = len(core_hits)

    is_hybrid = n_core_paths >= 2

    if n_core_paths == 0:
        locus_type = "none"
        label = None
    elif n_core_paths == 1:
        locus_type = "single"
        label = list(core_hits.keys())[0]
    else:
        locus_type = "hybrid"
        label = "+".join(sorted(core_hits.keys()))

    return {
        "pathways": results,
        "n_core_paths": n_core_paths,
        "is_hybrid": is_hybrid,
        "type": locus_type,
        "label": label
    }

In [9]:
# apply to df
loci["gene_set"] = loci["Gene"].str.split(',').apply(set)

loci["annotation"] = loci["gene_set"].apply(annotate_sulfo_pathways)

In [10]:
# filter for hybrids
loci[loci["annotation"].apply(lambda x: x["n_core_paths"] >= 2)]

,cluster,Gene,gene_set,annotation
1496,13517,"sqvA,sqvB,sqvD,sqwF,sqwG,sqwH,sqwI,yihQ","{sqvB, sqwH, sqwF, yihQ, sqvA, sqvD, sqwI, sqwG}","{'pathways': {'sulfo-TK': {'score': 'min_up', ..."
2675,23956,"sqvA,sqvB,sqwF,sqwG,sqwH,sqwI","{sqvB, sqwH, sqwF, sqvA, sqwI, sqwG}","{'pathways': {'sulfo-TK': {'score': 'min', 'co..."


In [11]:
# annotation expantion
expanded = pd.concat([
    pd.DataFrame([
        {
            "cluster": row.cluster,
            "SQ_pathway": p,
            "SQ_score": v["score"],
            "SQ_gene_content_set": ",".join(sorted(row.gene_set)),
            "SQ_label": row.annotation["label"],         
            "SQ_type": row.annotation["type"]             
        }
        for p, v in row.annotation["pathways"].items()
    ])
    for _, row in loci[
        loci["annotation"].apply(lambda x: len(x["pathways"]) > 0)
    ].iterrows()
], ignore_index=True)

# clusters metadata
meta = candidates.groupby("cluster").agg(
    contig=("contig", "first"),
    taxonomy=("taxonomy", "first"),
    target=("target", lambda x: ",".join(sorted(set(x)))),
    bitscore_list=("bitscore", lambda x: list(x)),
    evalue_list=("evalue", lambda x: list(x))
).reset_index()

# merge
final_table = expanded.merge(meta, on="cluster", how="left")

final_table = final_table[
    ["cluster", "target", "contig", "SQ_gene_content_set",
     "SQ_pathway", "SQ_score", "taxonomy", "bitscore_list", "evalue_list"]
]

In [12]:
# save
final_table.to_csv("../data/final_table.csv", index=False)

## Selecting the most complete clusters

In [13]:
df = pd.read_csv("../data/final_table.csv")

df = df[df["SQ_score"].isin(["min_up", "min_down", "max"])].copy()

df["locus"] = df["SQ_gene_content_set"].apply(
    lambda x: ",".join(sorted(x.split(",")))
)

df["order"] = df["taxonomy"].apply(
    lambda x: [t for t in x.split(";") if t.startswith("o__")][0].replace("o__", "")
)

df.to_csv("../data/clusters_filtered.csv", index=False)